# cyballs with an in-memory NumPy catalog

This example passes positions, scalar values, weights, and a mask directly to `cyballs`. No catalog file is created. The extension copies the arrays into C-owned body tables, so tree operations cannot modify the NumPy inputs.

In [ ]:
import tempfile
import numpy as np
from cyballs import cballs

## Build a catalog in memory

For the active 3D build, `positions` must have shape `(N, 3)`. Optional one-dimensional arrays must have length `N`.

In [ ]:
rng = np.random.default_rng(1729)
nbody = 128
positions = rng.uniform(-0.45, 0.45, size=(nbody, 3))
kappa = 1.0 + positions[:, 0] - 0.5 * positions[:, 1]
weights = rng.uniform(0.8, 1.2, size=nbody)
mask = np.ones(nbody, dtype=bool)
positions_before = positions.copy()

## Configure and register the catalog

Do not set `infile` or `infileformat` when using `set_catalog()`. For cross-correlation, set the normal `iCatalogs` parameter and call `set_catalog(..., catalog=1)` to register a second catalog after catalog 0.

In [ ]:
temporary_root = tempfile.TemporaryDirectory(prefix="cyballs-notebook-")
balls = cballs()
balls.set({
    "searchMethod": "octree-ggg-omp",
    "rangeN": 0.8,
    "rminHist": 0.02,
    "sizeHistN": 12,
    "mChebyshev": 3,
    "lengthBox": 1.0,
    "numberThreads": 2,
    "verbose": 0,
    "verbose_log": 0,
    "rootDir": temporary_root.name,
    "options": "no-out-Hist",
})
balls.set_catalog(positions, kappa=kappa, weights=weights, mask=mask)
print("registered catalogs:", balls.catalog_count)

## Run and read the live C results

In [ ]:
balls.Run(level=["MainLoop"])
radius = balls.getrBins().copy()
xi2pcf = balls.getHistXi2pcf().copy()
zeta_m1 = balls.getHistZetaMsincos(1, 1).copy()
print("C body count:", balls.getNBody())
print("r bins:", radius)
print("2PCF:", xi2pcf)
print("3PCF m=1 shape:", zeta_m1.shape)

## Release C storage

The registered NumPy catalog remains available for another `Run()` until `clear_catalogs()` or `set_default()` is called.

In [ ]:
balls.struct_cleanup()
np.testing.assert_array_equal(positions, positions_before)
temporary_root.cleanup()
print("C storage released; NumPy input unchanged.")